## RGB conversion of a batch of images

- This notebook takes as input the `water_detection_params.yaml`, where all the parameters that have been manually selected from `rgb_conversion_interactive.ipynb` have been written.
- The goal is to compute the baseline and automatically generate the final matrices with water mask and vegetation score for all the images in the considered batch, and then save these matrices in a specified folder.

## 1. Import Required Libraries and Modules
This cell imports all necessary Python libraries and project modules for image processing, configuration loading, and visualization. It also sets up the Python path and loads color maps for later use.

In [ ]:
import sys
import os
import yaml
import json
from pathlib import Path
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.exposure import match_histograms

cwd = Path.cwd().resolve()
for parent in [cwd, *cwd.parents]:
    if (parent / 'beaversim').is_dir():
        project_root = parent
        break
else:
    raise RuntimeError("Could not locate project root containing 'beaversim'.")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

try:
    from data_acquisition.modules.rgb_to_matrix import (
        load_rgb_image,
        calculate_ndvi,
        calculate_excess_green,
        calculate_cwi,
        detect_water_simple,
        apply_shadow_correction,
        postprocess_water_mask,
    )    
except Exception as e:    
    raise

try:
    from beaversim.ral.backend.modules.module_colors import ColorMaps
    colors = ColorMaps()    
except ImportError as e:    
    print("Warning: ColorMaps not available, using default matplotlib colormaps")
    colors = None
    
# Global variables
MAX_COLS = 3
VMIN = 0
VMAX = 1.5
MEAN = (VMAX + VMIN) / 2
print('✓ Modules loaded')

## 2. Load Configuration Parameters
This cell loads the processing parameters from a YAML configuration file, including paths, baseline method, shadow correction, and per-image settings. It prepares the configuration for all subsequent steps.

In [ ]:
# ============================================================================
# LOAD PARAMETERS FROM YAML CONFIG
# ============================================================================
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'config/water_detection_params_A2_annotated.yaml')

with open(CONFIG_PATH, 'r') as f:
    params = yaml.safe_load(f)

source_path = params['source_path'].replace('data_acquisition/', '../../data_acquisition/')
output_path = params['output_path'].replace('output/', '../../output/')

# ── General (shared) parameters ──────────────────────────────────────────────
general = params['pictures']['general']
baseline = params['baseline']

# BASELINE
BASELINE_METHOD      = str(baseline.get('method', 'median')).lower()
BASELINE_REMOVE_MEAN = baseline.get('remove_mean', False)
BASELINE_SHADOW_CORRECTION = baseline.get('shadow_correction', True)
BASELINE_WATER_INDEX          = baseline.get('water_index', 'NDVI')
BASELINE_WATER_THRESHOLD      = baseline.get('threshold', None)
BASELINE_OPENING_SIZES        = baseline.get('opening_sizes', [])
BASELINE_MEDIAN_SIZES         = baseline.get('median_sizes', [])
BASELINE_CLOSING_SIZES        = baseline.get('closing_sizes', [])
BASELINE_RESOLUTION         = baseline.get('resolution', 1.0)
BASELINE_DOWNSAMPLE_FACTOR = baseline.get('downsample_factor', 1)

# PICTURES
SHADOW_CORRECTION    = general.get('shadow_correction', general.get('shadow_correction', True))
VEGETATION_INDEX     = general.get('vegetation_index', 'NDVI')
REMOVE_MEAN        = general.get('remove_mean', False)

# ── Per-image config (skip the 'general' key) ────────────────────────────────
IMAGE_CONFIGS = {}
for pic_key, pic_val in params['pictures'].items():
    if pic_key == 'general':
        continue
    IMAGE_CONFIGS[pic_key] = {
        'name':       f"{pic_key} ({pic_val['image']})",
        'image_path': os.path.join(source_path, pic_val['image']),
        'output_dir': os.path.join(output_path, pic_key),
        # Allow per-image overrides, otherwise fall back to general values        
        'shadow_correction':  SHADOW_CORRECTION,
        'remove_mean':        REMOVE_MEAN,
        'vegetation_index':   VEGETATION_INDEX,        
    }

print("Baseline config:")
for k, v in baseline.items():
    print(f"  {k}: {v}")

print("Image configs:")
for k, v in general.items():
    print(f"  {k}: {v}")


## 3. Compute Baseline Image
This cell loads all images from the source folder, computes the mean and median RGB baselines, applies shadow correction, and visualizes the results. The baseline is used as a stable reference for water detection and normalization.

In [ ]:
# ============================================================================
# Load all images from GENERAL_FOLDER
# ============================================================================
SUPPORTED_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
image_paths = sorted([
    p for p in glob.glob(os.path.join(source_path, '*'))
    if p.lower().endswith(SUPPORTED_EXTS)
])

if not image_paths:
    raise FileNotFoundError(f"No images found in source_path: {source_path}")

print(f"Found {len(image_paths)} image(s) in: {source_path}")

# define the target size for resizing images (if needed)
min_height, min_width = None, None
for path in image_paths:
    with Image.open(path) as img:
        w, h = img.size
        if min_height is None or h < min_height:
            min_height = h
        if min_width is None or w < min_width:
            min_width = w

print(f"Smallest image size: height={min_height}, width={min_width}")
TARGET_SIZE = (min_width, min_height)

# Load all images, resize to the same size as the first one
ref_rgb, _ = load_rgb_image(image_paths[0], target_size=TARGET_SIZE)
H, W = ref_rgb.shape[:2]

# Stack all images into a 4D array (N, H, W, 3)
rgb_stack = np.zeros((len(image_paths), H, W, 3), dtype=np.float32)
for i, path in enumerate(image_paths):
    img, _ = load_rgb_image(path, target_size=TARGET_SIZE)
    rgb_stack[i] = img.astype(np.float32)

# Compute both so we can compare them
rgb_mean   = np.clip(rgb_stack.mean(axis=0), 0, 255).astype(np.uint8)
rgb_median = np.clip(np.median(rgb_stack, axis=0), 0, 255).astype(np.uint8)

# Apply shadow correction to the baseline image
if SHADOW_CORRECTION:
    rgb_mean_corrected = apply_shadow_correction(rgb_mean, method='clahe')
    rgb_median_corrected = apply_shadow_correction(rgb_median, method='clahe')
else:
    rgb_mean_corrected = rgb_mean.copy()
    rgb_median_corrected = rgb_median.copy()

# Select baseline according to chosen method
if BASELINE_METHOD == 'median':
    rgb_baseline = rgb_median_corrected
else:
    rgb_baseline = rgb_mean_corrected

print(f"\n✓ Baseline method: {BASELINE_METHOD.upper()}")

# ============================================================================
# Compare Mean vs Median side-by-side
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(rgb_mean_corrected)
axes[0].set_title('Mean\n(sensitive to outliers)', fontsize=13, fontweight='bold',
                  color='navy' if BASELINE_METHOD == 'mean' else 'black')
axes[0].axis('off')

axes[1].imshow(rgb_median_corrected)
axes[1].set_title('Median\n(robust to structural changes)', fontsize=13, fontweight='bold',
                  color='navy' if BASELINE_METHOD == 'median' else 'black')
axes[1].axis('off')

# Absolute per-pixel difference (averaged over channels, normalised for visibility)
diff = np.abs(rgb_mean_corrected.astype(np.int16) - rgb_median_corrected.astype(np.int16)).mean(axis=2)
diff_norm = (diff / diff.max() * 255).astype(np.uint8) if diff.max() > 0 else diff.astype(np.uint8)
im = axes[2].imshow(diff_norm, cmap='viridis')
axes[2].set_title('|Mean − Median|\n(bright = larger divergence)', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

selected_label = 'MEDIAN  ←  selected' if BASELINE_METHOD == 'median' else 'MEAN  ←  selected'
plt.suptitle(f'Mean vs Median Baseline Comparison  |  {selected_label}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Compute Vegetation Indices and Water Mask on Baseline
This cell calculates vegetation indices (NDVI, VARI, ExG, CWI) on the shadow-corrected baseline image, selects the water index, determines the threshold, detects water, and applies morphological post-processing. It visualizes the water detection results.

In [ ]:
# ============================================================================
# STEP 1: Compute Vegetation Indices on Baseline & Water Detection
# Indices are computed on rgb_baseline, (after shadow correction if applicable).
# The water index specified in the config (WATER_INDEX) is thresholded to
# produce a water mask, then cleaned up with morphological operations.
# ============================================================================

# ── Compute all indices on the baseline ──────────────────────────────────────
ndvi_baseline = calculate_ndvi(rgb_baseline, method='visible', remove_mean=BASELINE_REMOVE_MEAN)
exg_baseline  = calculate_excess_green(rgb_baseline, remove_mean=BASELINE_REMOVE_MEAN)
cwi_baseline  = calculate_cwi(rgb_baseline, remove_mean=BASELINE_REMOVE_MEAN)

index_map = {
    'NDVI': ndvi_baseline,    
    'ExG':  exg_baseline,
    'CWI':  cwi_baseline,
}

print('✓ Baseline indices computed (NDVI, VARI, ExG, CWI)')

# ── Select water index ────────────────────────────────────────────────────────
if BASELINE_WATER_INDEX not in index_map:
    raise ValueError(f"Unknown WATER_INDEX: '{BASELINE_WATER_INDEX}'. Choose from {list(index_map)}")
water_veg_index = index_map[BASELINE_WATER_INDEX]

# ── Determine threshold ───────────────────────────────────────────────────────
index_threshold = float(BASELINE_WATER_THRESHOLD)
print(f"  Using MANUAL threshold : {index_threshold:.3f}  ({BASELINE_WATER_INDEX})")

# ── Raw water detection ───────────────────────────────────────────────────────
water_mask_raw = detect_water_simple(
    rgb_baseline,
    water_veg_index,
    index_name=BASELINE_WATER_INDEX,
    index_threshold=index_threshold,
)

# ── Morphological post-processing ────────────────────────────────────────────
water_mask = postprocess_water_mask(
    water_mask_raw,
    opening_sizes=BASELINE_OPENING_SIZES,
    median_sizes=BASELINE_MEDIAN_SIZES,
    closing_sizes=BASELINE_CLOSING_SIZES,
    apply_opening=len(BASELINE_OPENING_SIZES) > 0,
    apply_median=len(BASELINE_MEDIAN_SIZES) > 0,
    apply_closing=len(BASELINE_CLOSING_SIZES) > 0,
)

water_pct = 100 * water_mask.sum() / water_mask.size
print(f"\n✓ Water mask ready  |  {water_pct:.2f}% water  ({water_mask.sum():,} px)")

# ── Visualize (2×2) ──────────────────────────────────────────────────────────
cmap_idx = plt.cm.RdYlGn.copy()
cmap_idx.set_under('dodgerblue')

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

vmin_index = 0
vmax_index = 1

# (0,0) Selected water index on baseline
im_idx = axes[0, 0].imshow(water_veg_index, cmap=cmap_idx, vmin=vmin_index, vmax=vmax_index)
axes[0, 0].set_title(f'Baseline {BASELINE_WATER_INDEX}\n(water detection index)', fontsize=13, fontweight='bold')
axes[0, 0].axis('off')
plt.colorbar(im_idx, ax=axes[0, 0], fraction=0.046, pad=0.04)

# (0,1) Raw water mask
axes[0, 1].imshow(water_mask_raw, cmap='Blues')
axes[0, 1].set_title(f'Raw Water Mask\n(threshold: {index_threshold:.3f})', fontsize=13, fontweight='bold')
axes[0, 1].axis('off')

# (1,0) Post-processed mask
axes[1, 0].imshow(water_mask, cmap='Blues')
axes[1, 0].set_title('Post-processed Water Mask\n(morphological cleanup)', fontsize=13, fontweight='bold')
axes[1, 0].axis('off')

# (1,1) Overlay on baseline
rgb_overlay = rgb_baseline.copy()
rgb_overlay[water_mask] = [30, 144, 255]
axes[1, 1].imshow(rgb_overlay)
axes[1, 1].set_title('Water Mask Overlay\n(on shadow-corrected baseline)', fontsize=13, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle(f'Water Detection on Baseline  |  Index: {BASELINE_WATER_INDEX}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Load, Shadow-Correct, and Mask All Images
This cell loads each image, applies shadow correction, resizes the water mask if needed, and paints water pixels blue. The processed images are stored for further analysis.

In [ ]:
# ============================================================================
# STEP 2: Load All Images, Shadow-Correct & Apply Water Mask
# For each image in IMAGE_CONFIGS:
#   - Load the original image
#   - Apply shadow correction
#   - Resize water_mask to match image resolution if needed
#   - Paint water pixels dodger-blue → rgb_masked
# Results are stored in processed_images[key] for use in Step 3.
# ============================================================================

processed_images = {}   # key → dict with rgb_original, rgb, rgb_masked

for key, cfg in IMAGE_CONFIGS.items():
    rgb_original, _ = load_rgb_image(cfg['image_path'], target_size=TARGET_SIZE)

    # Shadow correction
    if cfg['shadow_correction']:
        rgb = apply_shadow_correction(rgb_original, method='clahe')
    else:
        rgb = rgb_original.copy()

    # Paint water pixels dodger-blue
    rgb_masked = rgb.copy()
    rgb_masked[water_mask] = [30, 144, 255]

    processed_images[key] = {
        'rgb_original':       rgb_original,
        'rgb':                rgb,
        'water_mask': water_mask,
        'rgb_masked':         rgb_masked,
    }    

print(f"\n✓ All {len(processed_images)} images loaded and processed")


## 6. Histogram Matching for Weather Normalization
This cell matches the histogram of each image (land pixels only) to the shadow-corrected baseline, correcting for lighting differences. The result is used to ensure vegetation indices reflect real changes, not illumination artifacts.

In [ ]:
# ============================================================================
# STEP 3: Histogram Matching (Weather Normalisation)
# For each image, match its histogram to the shadow-corrected baseline using
# land-only pixels as the reference distribution.  The result (rgb_matched)
# corrects for variable lighting so that vegetation indices in Step 4 reflect
# genuine vegetation change rather than illumination artefacts.
# Adds 'rgb_matched' to each processed_images[key] entry.
# ============================================================================

base_h, base_w = rgb_baseline.shape[:2]

for key, entry in processed_images.items():
    rgb       = entry['rgb']
    land      = ~entry['water_mask']

    sel_h, sel_w = rgb.shape[:2]

    # Resize baseline to match image resolution if needed
    if (base_h, base_w) != (sel_h, sel_w):
        ref_for_match = np.array(
            Image.fromarray(rgb_baseline).resize((sel_w, sel_h), Image.BILINEAR)
        ).astype(np.uint8)
    else:
        ref_for_match = rgb_baseline

    # Per-channel matching using land pixels only as reference distribution
    rgb_matched = np.empty_like(rgb)
    for ch in range(3):
        src_flat = rgb[:, :, ch].astype(np.float32).ravel()
        ref_land = ref_for_match[:, :, ch].astype(np.float32)[land]   # 1-D land reference
        matched_flat = match_histograms(src_flat, ref_land)
        rgb_matched[:, :, ch] = np.clip(matched_flat.reshape(rgb.shape[:2]), 0, 255).astype(np.uint8)

    entry['ref_for_match'] = ref_for_match
    entry['rgb_matched']   = rgb_matched
    print(f"  ✓ {key:8s}  histogram matched")

print(f"\n✓ All {len(processed_images)} images histogram-matched")


## 7. Compute Vegetation Quality Score and Final Matrix
This cell computes the selected vegetation index for each image, compares it to the baseline, and calculates a quality score. Water pixels are set to -1. The results are visualized and stored for further analysis.

In [ ]:
# ============================================================================
# STEP 4: Vegetation Quality Score & Final Matrix
# For each image:
#   - Compute VEGETATION_INDEX on rgb_matched
#   - Score = (1 + index_selected) / (2 * (1 + index_baseline + EPS))
#   - Water pixels → -1
#   - Store final_matrix and save outputs to disk
# ============================================================================

EPS = 1.0   # stability epsilon for near-zero baseline values
FINAL_VALUE = 'index_selected'  # 'vegetation_score' or 'index_selected'

# Compute vegetation indices on the shadow-corrected baseline
print("\nComputing vegetation indices on baseline")
ndvi_baseline_land = calculate_ndvi(rgb_baseline, method='visible', remove_mean=BASELINE_REMOVE_MEAN, mask=water_mask, mean=MEAN)
exg_baseline_land  = calculate_excess_green(rgb_baseline, remove_mean=BASELINE_REMOVE_MEAN, mask=water_mask, mean=MEAN)
cwi_baseline_land  = calculate_cwi(rgb_baseline, remove_mean=BASELINE_REMOVE_MEAN, mask=water_mask, mean=MEAN)

# Baseline index (computed in Step 1 on rgb_baseline_corrected)
index_map = {
    'NDVI': ndvi_baseline_land,    
    'ExG':  exg_baseline_land,
    'CWI':  cwi_baseline_land,
}
if VEGETATION_INDEX not in index_map:
    raise ValueError(f"Unknown VEGETATION_INDEX '{VEGETATION_INDEX}'. Choose from {list(index_map)}")
index_baseline = index_map[VEGETATION_INDEX]

index_fn_map = {
    'NDVI': lambda img: calculate_ndvi(img, method='visible', remove_mean=IMAGE_CONFIGS[key]['remove_mean'], mask=entry['water_mask'], mean=MEAN),    
    'ExG':  lambda img: calculate_excess_green(img, remove_mean=IMAGE_CONFIGS[key]['remove_mean'], mask=entry['water_mask'], mean=MEAN),
    'CWI':  lambda img: calculate_cwi(img, remove_mean=IMAGE_CONFIGS[key]['remove_mean'], mask=entry['water_mask'], mean=MEAN),
}
compute_index = index_fn_map[VEGETATION_INDEX]

cmap_idx   = plt.cm.RdYlGn.copy()
cmap_idx.set_bad('dodgerblue')
cmap_score = plt.cm.RdYlGn.copy()
cmap_score.set_bad('dodgerblue') 

for key, entry in processed_images.items():
    rgb_matched = entry['rgb_matched']
    wm          = entry['water_mask']
    LAND        = ~wm
    
    print(f"\nProcessing {key}  |  Vegetation index: {VEGETATION_INDEX}")
    index_selected = compute_index(rgb_matched).astype(np.float32)
    
    if FINAL_VALUE == 'index_selected':
        # --- Compute vegetation index on matched image ---
        final_matrix = index_selected.copy()
        baseline_matrix = index_baseline.copy()
    else:
        # --- Score: (1 + index_selected) / (2 * (1 + index_baseline + eps)) → [0, 1] ---
        vegetation_score = np.full_like(index_selected, fill_value=np.nan, dtype=np.float32)
        vegetation_score[LAND] = (1.0 + index_selected[LAND] + EPS) / (2.0 * (1.0 + index_baseline[LAND] + EPS))    

        # --- Baseline image: INDEX on land, -1 on water ---
        baseline_score = np.full_like(index_baseline, fill_value=np.nan, dtype=np.float32)
        baseline_score[LAND] = (1.0 + index_baseline[LAND] + EPS) / (2.0 * (1.0 + index_baseline[LAND] + EPS))
        baseline_matrix = baseline_score.copy()    
        
        final_matrix = vegetation_score.copy()
        baseline_matrix = baseline_score.copy()
    
        # Water pixels → -1
        print(f"✓ Vegetation score computed using {VEGETATION_INDEX} for {key}")
        print(f"  Land score range : [{vegetation_score[LAND].min():.3f}, {vegetation_score[LAND].max():.3f}]")
        print(f"  Mean land score  : {vegetation_score[LAND].mean():.3f}")
        
    # save the baseline and final matrices to the processed_images dict for later use
    entry['baseline_matrix'] = baseline_matrix
    entry['final_matrix'] = final_matrix


    # ── Visualisation: 2×2 grid per image ────────────────────────────────────        
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # (0,0) Baseline raw index
    im00 = axes[0].imshow(baseline_matrix, cmap=cmap_idx, vmin=VMIN, vmax=VMAX)
    axes[0].set_title(f'Baseline {VEGETATION_INDEX} (raw index)', fontsize=13, fontweight='bold')
    axes[0].axis('off')
    cbar = plt.colorbar(im00, ax=axes[0], fraction=0.046, pad=0.04, shrink=0.7)
    cbar.ax.tick_params(labelsize=16)
    cbar.ax.set_ylabel(f'{VEGETATION_INDEX}', fontsize=20)

    # (0,1) Final quality matrix
    im01 = axes[1].imshow(final_matrix, cmap=cmap_score, vmin=VMIN, vmax=VMAX)
    axes[1].set_title(f'Final Quality Matrix\n(land: [0,1] | water: −1)', fontsize=13, fontweight='bold')
    axes[1].axis('off')
    cbar = plt.colorbar(im01, ax=axes[1], fraction=0.046, pad=0.04, shrink=0.7)
    cbar.ax.tick_params(labelsize=16)
    cbar.ax.set_ylabel(f'{VEGETATION_INDEX}', fontsize=20)

    plt.suptitle(f'Vegetation Quality Matrix  |  Index: {VEGETATION_INDEX}\n'
                'Top: raw index comparison  |  Bottom: normalised scores',
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

def plot_grid_on_baseline(baseline, n, ax=None):
    """
    Show the baseline image with an n x n grid overlay and label each cell.
    Cell numbers are row-major, starting from 1 (top-left).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(baseline)
    h, w = baseline.shape[:2]
    # Draw grid lines
    for i in range(1, n):
        ax.axhline(i * h // n, color='red', lw=2, linestyle='--')
        ax.axvline(i * w // n, color='red', lw=2, linestyle='--')
    # Add cell numbers
    for r in range(n):
        for c in range(n):
            idx = r * n + c + 1
            y0, y1 = r * h // n, (r + 1) * h // n if r < n - 1 else h
            x0, x1 = c * w // n, (c + 1) * w // n if c < n - 1 else w
            cy = (y0 + y1) // 2
            cx = (x0 + x1) // 2
            ax.text(cx, cy, str(idx), color='white', fontsize=18, fontweight='bold',
                    ha='center', va='center',
                    bbox=dict(facecolor='red', alpha=0.5, boxstyle='circle,pad=0.2'))
    ax.set_title(f'Baseline with {n}x{n} grid')
    ax.axis('off')

def compute_spatial_pearsons(processed_images, n_cells_list=[1, 4, 9, 16], baseline_img=None):
    image_keys = sorted(processed_images.keys())
    n_images = len(image_keys)
    matrix_shape = processed_images[image_keys[0]]['final_matrix'].shape        

    # --- Show all final matrices in a single row before the Pearson plots ---
    cmap_idx = plt.cm.RdYlGn.copy()
    cmap_idx.set_bad('dodgerblue')
    fig, axes = plt.subplots(1, n_images + 1, figsize=(6 * n_images + 1, 4))
    
    mat = processed_images[image_keys[0]]['baseline_matrix']    
    im = axes[0].imshow(mat, cmap=cmap_idx, vmin=VMIN, vmax=VMAX)
    axes[0].set_title('Baseline', fontsize=12)
    axes[0].axis('off')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04).set_label('Score', fontsize=10)
    
    if n_images == 1:
        axes = [axes]
    for i, key in enumerate(image_keys):
        tmp = processed_images[key]['final_matrix']   
        mat = np.copy(tmp)            
        im = axes[i+1].imshow(mat, cmap=cmap_idx, vmin=VMIN, vmax=VMAX)
        axes[i+1].set_title(key, fontsize=12)
        axes[i+1].axis('off')
        plt.colorbar(im, ax=axes[i+1], fraction=0.046, pad=0.04).set_label('Score', fontsize=10)
    plt.suptitle('All Final Matrices (one row)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    for n_cells in n_cells_list:
        n = int(np.sqrt(n_cells))
        assert n * n == n_cells, "n_cells must be a perfect square (4, 9, 16, ...)"
        h, w = matrix_shape
        h_step, w_step = h // n, w // n

        pearson_grid = [[] for _ in range(n_cells + 1)]  # +1 for full image
        pair_labels = []

        for i in range(n_images):
            key1 = image_keys[i]
            mat1 = processed_images[key1]['baseline_matrix']      
            # mat1 = processed_images[image_keys[0]]['final_matrix']      
            tmp = processed_images[key1]['final_matrix']                        
            mat2 = np.copy(tmp)            
            
            pair_labels.append(f"base-{key1}")

            # Full image Pearson
            land_mask_full = (mat1 > -1) & (mat2 > -1)
            if np.any(land_mask_full):
                corr_full, _ = pearsonr(mat1[land_mask_full].flatten(), mat2[land_mask_full].flatten())                
            else:
                corr_full = np.nan
            pearson_grid[0].append(corr_full)

            # Grid cells
            for r in range(n):
                for c in range(n):
                    idx = r * n + c + 1  # +1 to leave 0 for full image
                    y0, y1 = r * h_step, (r + 1) * h_step if r < n - 1 else h
                    x0, x1 = c * w_step, (c + 1) * w_step if c < n - 1 else w
                    zone1 = mat1[y0:y1, x0:x1]
                    zone2 = mat2[y0:y1, x0:x1]
                    land_mask = (zone1 > -1) & (zone2 > -1)
                    if np.any(land_mask):
                        corr, _ = pearsonr(zone1[land_mask].flatten(), zone2[land_mask].flatten())                        
                    else:
                        corr = np.nan
                    pearson_grid[idx].append(corr)        

        # Plot Pearson correlations and grid overlay side by side
        fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [2, 1]})
        ax = axes[0]
        # Use a colormap for the grid zones (skip the first color for full image)
        cmap = plt.get_cmap('tab20', n_cells + 1)
        # Plot full image correlation in black and bold
        ax.plot(pair_labels, pearson_grid[0], marker='o', label='full_image', color='black', linewidth=2)
        # Plot each zone with a unique color and legend as 'cell N'
        for idx, vals in enumerate(pearson_grid[1:]):
            color = cmap(idx + 1)
            label = f'cell {idx+1}'
            ax.plot(pair_labels, vals, marker='o', label=label, color=color)
        ax.set_title(f"Spatial Pearson Correlation ({n}x{n} grid + full image)")
        ax.set_xlabel("Image Pair")
        ax.set_ylabel("Pearson r")
        # ax.set_ylim(-1, 1)
        ax.legend(title="Cell number", bbox_to_anchor=(1.05, 1), loc='upper left', ncol=1)
        ax.grid(True)

        # Show grid overlay on baseline image
        if baseline_img is not None:
            plot_grid_on_baseline(baseline_img, n, ax=axes[1])
        else:
            axes[1].axis('off')
        plt.tight_layout()
        plt.show()

# Example usage:
# Pass the baseline image (e.g., rgb_baseline) as baseline_img
compute_spatial_pearsons(processed_images, n_cells_list=[4, 9], baseline_img=rgb_baseline)


In [ ]:
# Output directory (ensure it exists)
output_dir = os.path.join(output_path)
os.makedirs(output_dir, exist_ok=True)

# baseline to be saved
baseline_matrix_save = baseline_matrix.copy()
baseline_matrix_save[water_mask] = -1  # Ensure water pixels are set to -1 in the saved baseline

# Determine downsampling factor
ds = int(BASELINE_DOWNSAMPLE_FACTOR)
if ds < 1:
    ds = 1

# Downsample baseline if needed
if ds > 1:
    baseline_matrix_save_ds = baseline_matrix_save[::ds, ::ds]
else:
    baseline_matrix_save_ds = baseline_matrix_save

# Determine the spatial coordinates for the downsampled baseline matrix
height, width = baseline_matrix_save_ds.shape
x_min = 0  # or set to your actual starting x
y_min = 0  # or set to your actual starting y

# Effective resolution after downsampling
base_resolution = float(BASELINE_RESOLUTION)
effective_resolution = base_resolution * ds

x_new = np.arange(x_min, x_min + width * effective_resolution, effective_resolution)
y_new = np.arange(y_min, y_min + height * effective_resolution, effective_resolution)

X_new, Y_new = np.meshgrid(x_new, y_new)

# Mirror for saving
X_new_mirrored = np.flipud(X_new)
Y_new_mirrored = np.flipud(Y_new)
baseline_matrix_mirrored = np.flipud(baseline_matrix_save_ds)

# Estimate actual spacing from generated coordinate grids
est_dx = float(np.nanmedian(np.diff(X_new_mirrored, axis=1))) if width > 1 else float('nan')
est_dy = float(np.nanmedian(np.abs(np.diff(Y_new_mirrored, axis=0)))) if height > 1 else float('nan')

# Save coordinate matrices
x_coords_path = os.path.join(output_dir, "X_coordinates.npy")
y_coords_path = os.path.join(output_dir, "Y_coordinates.npy")
np.save(x_coords_path, X_new_mirrored)
np.save(y_coords_path, Y_new_mirrored)
print(f"✓ Coordinates saved: {x_coords_path}, {y_coords_path}")
print(f"✓ Estimated output resolution (x, y): ({est_dx:.3f}, {est_dy:.3f})")

# Save baseline matrix
baseline_matrix_path = os.path.join(output_dir, "baseline_matrix.npy")
np.save(baseline_matrix_path, baseline_matrix_mirrored)
print(f"✓ Baseline matrix saved: {baseline_matrix_path}")

# Save final matrix for each image and collect metadata
processing_metadata = {
    "baseline_matrix": "baseline_matrix.npy",
    "x_coordinates": "X_coordinates.npy",
    "y_coordinates": "Y_coordinates.npy",
    "base_resolution": base_resolution,
    "downsample_factor": ds,
    "effective_resolution": effective_resolution,
    "estimated_resolution_x": est_dx if np.isfinite(est_dx) else None,
    "estimated_resolution_y": est_dy if np.isfinite(est_dy) else None,
    "baseline_shape": list(baseline_matrix_mirrored.shape),
    "images": []
}

for key, entry in processed_images.items():
    matrix_path = os.path.join(output_dir, f"{key}_final_matrix.npy")
    final_matrix = entry["final_matrix"]
    final_matrix[water_mask] = -1
    # Downsample
    if ds > 1:
        final_matrix_ds = final_matrix[::ds, ::ds]
    else:
        final_matrix_ds = final_matrix
    final_matrix_mirrored = np.flipud(final_matrix_ds)
    np.save(matrix_path, final_matrix_mirrored)
    print(f"✓ Final matrix saved for {key}: {matrix_path}")
    matrix = final_matrix_mirrored
    land_mask = matrix > -1
    land_mean = float(np.nanmean(matrix[land_mask])) if np.any(land_mask) else float('nan')
    processing_metadata["images"].append({
        "key": key,
        "matrix_file": f"{key}_final_matrix.npy",
        "original_image": os.path.basename(entry["rgb_original"]) if isinstance(entry["rgb_original"], str) else None,
        "shape": matrix.shape,
        "min": float(np.nanmin(matrix)),
        "max": float(np.nanmax(matrix)),
        "mean_land": land_mean
    })

# Save processing metadata
metadata_path = os.path.join(output_dir, "processing_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(processing_metadata, f, indent=2)
print(f"✓ Processing metadata saved: {metadata_path}")